In [ ]:
import requests
from datetime import datetime, timedelta

# Buscar dados de terremotos de 01/Jan/2024 até 01/Jan/2026 (em lotes mensais)
start_date = datetime(2024, 1, 1)
end_date = datetime(2026, 1, 1)

all_features = []
current = start_date

while current < end_date:
    next_date = current + timedelta(days=30)
    if next_date > end_date:
        next_date = end_date
    
    url = f"https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime={current.strftime('%Y-%m-%d')}&endtime={next_date.strftime('%Y-%m-%d')}&minmagnitude=2.5"
    
    response = requests.get(url)
    if response.status_code == 200:
        batch = response.json()
        all_features.extend(batch["features"])
        print(f"  {current.strftime('%Y-%m-%d')} a {next_date.strftime('%Y-%m-%d')}: {batch['metadata']['count']} registros")
    else:
        print(f"  ERRO {response.status_code} para {current.strftime('%Y-%m-%d')}")
    
    current = next_date

data = {"features": all_features}
print(f"\nTotal de terremotos coletados: {len(all_features)}")

In [ ]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, IntegerType

# Extrair features do JSON
features = data["features"]

# Transformar em lista de dicionários estruturados
rows = []
for f in features:
    props = f["properties"]
    coords = f["geometry"]["coordinates"]
    rows.append(Row(
        id=f["id"],
        magnitude=float(props["mag"]) if props["mag"] is not None else None,
        place=props["place"],
        time=props["time"],
        updated=props["updated"],
        tsunami=props["tsunami"],
        sig=props["sig"],
        net=props["net"],
        mag_type=props["magType"],
        event_type=props["type"],
        title=props["title"],
        longitude=float(coords[0]),
        latitude=float(coords[1]),
        depth=float(coords[2])
    ))

# Criar DataFrame e salvar como tabela Delta
df = spark.createDataFrame(rows)
df.write.mode("overwrite").saveAsTable("terremoto_api.default.earthquakes_usgs")

print(f"Tabela salva com {df.count()} registros.")
display(df)

In [ ]:
%pip install azure-storage-blob -q

In [ ]:
from azure.storage.blob import BlobServiceClient
import tempfile
import os
import pandas as pd

# Configurações do Storage Account
account_name = "claytonmedeiros1"
account_key = "oBJbZ8zcrzAvvbdIROytVL22vKN7VCS2UDB7BOrgDFXi8av9F+ll5XCRavUwCE6yYCo4cHzX9OyA+ASt2RDubg=="
container_name = "teste1"

# Ler da tabela Delta e converter para Pandas
pdf = spark.table("terremoto_api.default.earthquakes_usgs").toPandas()
pdf_clean = pd.DataFrame(pdf.to_dict())

# Criar colunas de partição (ano/mes/dia) a partir do timestamp em milissegundos
pdf_clean["event_date"] = pd.to_datetime(pdf_clean["time"], unit="ms")
pdf_clean["ano"] = pdf_clean["event_date"].dt.year
pdf_clean["mes"] = pdf_clean["event_date"].dt.month
pdf_clean["dia"] = pdf_clean["event_date"].dt.day
pdf_clean.drop(columns=["event_date"], inplace=True)

# Upload particionado por ano/mes/dia
connection_string = f"DefaultEndpointsProtocol=https;AccountName={account_name};AccountKey={account_key};EndpointSuffix=core.windows.net"
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

uploaded = 0
for (ano, mes, dia), group in pdf_clean.groupby(["ano", "mes", "dia"]):
    # Salvar cada partição como parquet
    tmp_path = os.path.join(tempfile.gettempdir(), f"earthquakes_{ano}_{mes:02d}_{dia:02d}.parquet")
    group.drop(columns=["ano", "mes", "dia"]).to_parquet(tmp_path, index=False)

    # Caminho no blob: earthquakes_usgs/ano=YYYY/mes=MM/dia=DD/data.parquet
    blob_path = f"earthquakes_usgs/ano={ano}/mes={mes:02d}/dia={dia:02d}/data.parquet"
    blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_path)

    with open(tmp_path, "rb") as data_file:
        blob_client.upload_blob(data_file, overwrite=True)
    
    uploaded += len(group)
    print(f"  Salvo: {blob_path} ({len(group)} registros)")

print(f"\nUpload concluído! Total: {uploaded} registros particionados em https://{account_name}.blob.core.windows.net/{container_name}/earthquakes_usgs/")